In [ ]:
import kagglehub
import os
import pandas as pd
import cv2

In [ ]:
path = "./data"
print("Path to dataset files : ",path)

In [ ]:
# Paths to the CSV files
train_csv_path = os.path.join(path, 'train.csv')
test_csv_path = os.path.join(path, 'test.csv')

# Path to the folder containing the actual images
# Based on the image, it looks like 'faces-spring-2020' is the image directory
image_dir = os.path.join(path, 'faces')

# Load the training data to verify the path
df_train = pd.read_csv(train_csv_path)
df_train['id'] = df_train['id'].astype(int).astype(str)
df_train['id'] = 'face-'+df_train['id']+'.png'

df_test = pd.read_csv(test_csv_path)
df_test['id'] = df_test['id'].astype(int).astype(str)
df_test['id'] = 'face-'+df_test['id']+'.png'

print(f"Total images found in train.csv: {len(df_train)}")
print(f"Total images found in train.csv: {len(df_test)}")

In [ ]:
final_dim = 64
processed_dir = path + "/faces_processed_" + str(final_dim)
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)


In [ ]:
import os
import cv2
import pandas as pd

# Define your interval (e.g., every 50 images)
progress_interval = 50 
count = 0

# Correcting the concat syntax: it needs a list [df_train, df_test]
combined_df = pd.concat([df_train, df_test])
total_images = len(combined_df)

print(f"Resizing images to {final_dim}x{final_dim}...")

for index, row in combined_df.iterrows():
    filename = row['id']
    input_path = os.path.join(image_dir, filename)
    output_path = os.path.join(processed_dir, filename)
    
    # 1. Check if the file already exists in the output directory
    if os.path.exists(output_path):
        count += 1
        continue # Skip to the next image
    
    img = cv2.imread(input_path)
    
    if img is not None:
        resized_img = cv2.resize(img, (final_dim, final_dim), interpolation=cv2.INTER_AREA)
        cv2.imwrite(output_path, resized_img)
        count += 1
        
        # 2. Progress indicator with custom counter
        if count % progress_interval == 0 or count == total_images:
            print(f"Progress: {count}/{total_images} images processed.")
    else:
        print(f"Warning: Could not read {filename} at {input_path}")
        count += 1

print(f"Successfully saved resized images to {processed_dir}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

# Ensure this matches your column name (e.g., 'glasses' or 'has_glasses')
label_col = 'glasses' 

def easy_correct(df, batch_size=12):
    current_idx = 0
    print("--- Data Correction Mode ---")
    print("Instructions:")
    print("- Press ENTER to see the next batch.")
    print("- Type an index (e.g., '42') to flip its label.")
    print("- Type 'stop' to finish and save.")

    while current_idx < len(df):
        plt.figure(figsize=(15, 10))
        batch_indices = range(current_idx, min(current_idx + batch_size, len(df)))
        
        for i, idx in enumerate(batch_indices):
            row = df.iloc[idx]
            img_path = os.path.join(processed_dir, row['id'])
            img = cv2.imread(img_path)
            
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                plt.subplot(3, 4, i + 1)
                plt.imshow(img)
                plt.title(f"Idx: {idx} | Label: {row[label_col]}")
                plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        user_cmd = input("Action (Index to flip / Enter for next / 'stop'): ").strip().lower()
        
        if user_cmd == 'stop':
            break
        elif user_cmd.isdigit():
            target_idx = int(user_cmd)
            # Flip the label: 0 becomes 1, 1 becomes 0
            old_val = df.at[target_idx, label_col]
            df.at[target_idx, label_col] = 1 - old_val
            print(f"!!! Corrected Index {target_idx}: {old_val} -> {df.at[target_idx, label_col]}")
            # Do not increment current_idx so you can see the change in the same batch
            continue 
        
        current_idx += batch_size

# Run the tool
easy_correct(combined_df)

In [1]:
from models.vae import VAE
import torch

# 1. Define the device (RTX 3060)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Initialize the model with your desired hyperparameters
# This is where you can change latent_dim or kernel_size for your ablation table
model = VAE(latent_dim=128, kernel_size=3).to(device)

# 3. Now run your dummy test
dummy_input = torch.randn(4, 3, 64, 64).to(device)
recon_batch, mu, logvar = model(dummy_input)

print(f"Success! Output shape: {recon_batch.shape}")

Success! Output shape: torch.Size([4, 3, 64, 64])


In [2]:
from models.vae import vae_loss_function
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for batch_idx, (data, _) in enumerate(train_loader):
    data = data.to(device)
    optimizer.zero_grad()
    
    recon_batch, mu, logvar = model(data)
    loss = vae_loss_function(recon_batch, data, mu, logvar)
    
    loss.backward()
    optimizer.step()
    
    if batch_idx % 10 == 0:
        print(f"Loss: {loss.item():.4f}")
    if batch_idx > 50: break # Just a quick test

NameError: name 'train_loader' is not defined